# A100 40 GB historical-protocol reproduction

Author: Mehran Keivanimehr.

SCIENTIFIC STATUS: this reproduces the historical protocol, including measurement limitations. It is not the corrected validation protocol. Read CURRENT_REPORT.md and the project scientific status register before interpreting training updates, TTFT, energy, capacity or MFU. New A100 experimental validation remains pending.

Use an exclusive Colab A100 runtime. The recorded main stack was torch 2.11.0+cu130, Python 3.13.15, Transformers 5.18.0, Accelerate 1.15.0. Colab's base image and unrecorded transitive dependencies may differ. This notebook has not been GPU-rerun during repository preparation.

Run in order. Outputs go to a new local directory. The Qwen dtype audit is retained. The vLLM setup downloads a separate environment and model weights. Historical cleanup, retry, account authentication, and remote upload cells are excluded.


In [ ]:
import subprocess, torch, json, platform
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,driver_version,power.limit","--format=csv"],capture_output=True,text=True).stdout)
p = torch.cuda.get_device_properties(0)
print(json.dumps({"gpu": p.name, "vram_gb": round(p.total_memory/1e9,1), "capability": f"{p.major}.{p.minor}",
  "bf16": torch.cuda.is_bf16_supported(), "torch": torch.__version__, "cuda": torch.version.cuda,
  "python": platform.python_version()}, indent=2))


In [ ]:
# Block 1 — environment, results folder, env.json
%pip install -q transformers==5.18.0 accelerate==1.15.0 nvidia-ml-py pandas tabulate

import os, json, platform, subprocess, datetime
import torch, transformers, accelerate

RESULTS_DIR = '/content/gpu_bench/a100_reproduction'
os.makedirs(RESULTS_DIR, exist_ok=True)

p = torch.cuda.get_device_properties(0)
smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version,power.limit,clocks.max.sm",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
env = {
    "gpu": p.name, "vram_gb": round(p.total_memory / 2**30, 1), "capability": f"{p.major}.{p.minor}",
    "sm_count": p.multi_processor_count, "nvidia_smi": smi,
    "torch": torch.__version__, "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
    "transformers": transformers.__version__, "accelerate": accelerate.__version__,
    "python": platform.python_version(), "cpu_count": os.cpu_count(),
    "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
}
json.dump(env, open(f"{RESULTS_DIR}/env.json", "w"), indent=2)
print(json.dumps(env, indent=2))
print("Results ->", RESULTS_DIR)


In [ ]:
# Block 2 — shared helpers: GPU sampler, timing, resumable result saving
import time, threading, csv, gc, contextlib
import pynvml

pynvml.nvmlInit()
_H = pynvml.nvmlDeviceGetHandleByIndex(0)
GPU_NAME = torch.cuda.get_device_name(0)

# A100 dense peak TFLOPS, used for an approximate MFU column
PEAK_TFLOPS = {"fp32": 19.5, "tf32": 156.0, "fp16": 312.0, "bf16": 312.0}

class GPUSampler:
    """Samples util / power / memory every `interval` s in a background thread."""
    def __init__(self, interval=0.1):
        self.interval, self.util, self.power, self.mem = interval, [], [], []
        self._stop = threading.Event()
    def _run(self):
        while not self._stop.is_set():
            self.util.append(pynvml.nvmlDeviceGetUtilizationRates(_H).gpu)
            self.power.append(pynvml.nvmlDeviceGetPowerUsage(_H) / 1000)
            self.mem.append(pynvml.nvmlDeviceGetMemoryInfo(_H).used / 2**30)
            time.sleep(self.interval)
    def __enter__(self):
        self._t = threading.Thread(target=self._run, daemon=True); self._t.start(); return self
    def __exit__(self, *a):
        self._stop.set(); self._t.join()
    def stats(self):
        avg = lambda x: round(sum(x) / len(x), 1) if x else None
        return {"gpu_util_avg": avg(self.util), "gpu_util_max": max(self.util, default=None),
                "power_avg_w": avg(self.power), "power_max_w": round(max(self.power, default=0), 1),
                "nvml_mem_max_gb": round(max(self.mem, default=0), 2)}

def cleanup():
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

def set_tf32(on):
    torch.set_float32_matmul_precision("high" if on else "highest")
    torch.backends.cudnn.allow_tf32 = on

def amp_ctx(prec):
    if prec == "fp16": return torch.autocast("cuda", dtype=torch.float16)
    if prec == "bf16": return torch.autocast("cuda", dtype=torch.bfloat16)
    return contextlib.nullcontext()          # fp32 / tf32

def run_timed(step_fn, warmup, iters):
    """Warm up, then time `iters` steps. Returns seconds, GPU stats, peak alloc/reserved GB."""
    for _ in range(warmup): step_fn()
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    with GPUSampler() as s:
        t0 = time.perf_counter()
        for _ in range(iters): step_fn()
        torch.cuda.synchronize()
        dt = time.perf_counter() - t0
    return (dt, s.stats(),
            round(torch.cuda.max_memory_allocated() / 2**30, 2),
            round(torch.cuda.max_memory_reserved() / 2**30, 2))

def _path(name): return f"{RESULTS_DIR}/{name}.jsonl"

def done_keys(name):
    if not os.path.exists(_path(name)): return set()
    return {json.loads(l)["key"] for l in open(_path(name)) if l.strip()}

def save_result(name, row):
    """Append to JSONL (fsync'd), then rewrite the matching CSV."""
    with open(_path(name), "a") as f:
        f.write(json.dumps(row) + "\n"); f.flush(); os.fsync(f.fileno())
    rows = [json.loads(l) for l in open(_path(name)) if l.strip()]
    cols = list(dict.fromkeys(k for r in rows for k in r))
    with open(f"{RESULTS_DIR}/{name}.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols); w.writeheader(); w.writerows(rows)

def base_row(key, **kw):
    return {"key": key, "gpu": GPU_NAME, "torch": torch.__version__,
            "ts": datetime.datetime.now().isoformat(timespec="seconds"), **kw}

with GPUSampler() as s: time.sleep(0.5)
print("Sampler OK:", s.stats())


In [ ]:
# Block 3 — GPT-2 124M training (synthetic tokens, seq 1024, AdamW)
from transformers import GPT2Config, GPT2LMHeadModel
import pandas as pd

WARMUP, ITERS, SEQ = 10, 40, 1024
PRECISIONS = ["fp32", "tf32", "fp16", "bf16"]   # fp32 = strict IEEE (TF32 off)
BATCHES = [8, 16, 32]

def gpt2_train(prec, bs):
    set_tf32(prec != "fp32")
    torch.manual_seed(0)
    model = GPT2LMHeadModel(GPT2Config(n_positions=SEQ)).cuda().train()
    n_params = sum(p.numel() for p in model.parameters())
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4, fused=True)
    scaler = torch.amp.GradScaler("cuda", enabled=(prec == "fp16"))
    x = torch.randint(0, model.config.vocab_size, (bs, SEQ), device="cuda")

    def step():
        with amp_ctx(prec):
            loss = model(input_ids=x, labels=x).loss
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update()
        opt.zero_grad(set_to_none=True)

    dt, gpu, peak_alloc, peak_res = run_timed(step, WARMUP, ITERS)
    tok_s = bs * SEQ * ITERS / dt
    tflops = 6 * n_params * tok_s / 1e12          # approx, excludes attention FLOPs
    return {"status": "ok", "tokens_per_s": round(tok_s), "step_ms": round(1000 * dt / ITERS, 1),
            "peak_alloc_gb": peak_alloc, "peak_reserved_gb": peak_res, **gpu,
            "tokens_per_joule": round(tok_s / gpu["power_avg_w"], 1) if gpu["power_avg_w"] else None,
            "approx_tflops": round(tflops, 1), "approx_mfu_pct": round(100 * tflops / PEAK_TFLOPS[prec], 1),
            "n_params_m": round(n_params / 1e6, 1), "attn_impl": model.config._attn_implementation}

NAME = "train_gpt2"
done = done_keys(NAME)
for prec in PRECISIONS:
    for bs in BATCHES:
        key = f"gpt2-124m|{prec}|bs{bs}|seq{SEQ}"
        if key in done:
            print("skip (done):", key); continue
        row = base_row(key, model="gpt2-124m", task="train", precision=prec, batch=bs, seq_len=SEQ,
                       warmup=WARMUP, iters=ITERS, optimizer="AdamW(fused)", data="synthetic")
        try:
            row.update(gpt2_train(prec, bs))
        except torch.cuda.OutOfMemoryError:
            row.update(status="OOM")
        cleanup()
        save_result(NAME, row)
        print(f"{key:32s} {row['status']:4s} "
              f"{row.get('tokens_per_s', '-'):>8} tok/s  peak {row.get('peak_alloc_gb', '-')} GB  "
              f"util {row.get('gpu_util_avg', '-')}%  {row.get('power_avg_w', '-')} W")

df = pd.read_csv(f"{RESULTS_DIR}/{NAME}.csv")
print(df[["precision", "batch", "status", "tokens_per_s", "step_ms", "peak_alloc_gb",
          "gpu_util_avg", "power_avg_w", "tokens_per_joule", "approx_mfu_pct"]].to_string(index=False))


In [ ]:
# Block 4 — ResNet-50 training (synthetic 224x224, channels_last, SGD momentum)
import torchvision

print(pd.read_csv(f"{RESULTS_DIR}/train_gpt2.csv").query("status == 'ok'")[["precision", "attn_impl"]]
      .drop_duplicates().to_string(index=False), "\n")

WARMUP, ITERS, IMG = 10, 40, 224
RN_BATCHES = [64, 128, 256]
RN_TRAIN_FLOPS_PER_IMG = 3 * 2 * 4.1e9      # ~4.1 GMACs fwd; train ≈ 3x fwd (approx)
torch.backends.cudnn.benchmark = True

def resnet_train(prec, bs):
    set_tf32(prec != "fp32")
    torch.manual_seed(0)
    model = torchvision.models.resnet50(weights=None).cuda().to(memory_format=torch.channels_last).train()
    opt = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4)
    scaler = torch.amp.GradScaler("cuda", enabled=(prec == "fp16"))
    loss_fn = torch.nn.CrossEntropyLoss()
    x = torch.randn(bs, 3, IMG, IMG, device="cuda").to(memory_format=torch.channels_last)
    y = torch.randint(0, 1000, (bs,), device="cuda")

    def step():
        with amp_ctx(prec):
            loss = loss_fn(model(x), y)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update()
        opt.zero_grad(set_to_none=True)

    dt, gpu, peak_alloc, peak_res = run_timed(step, WARMUP, ITERS)
    img_s = bs * ITERS / dt
    tflops = RN_TRAIN_FLOPS_PER_IMG * img_s / 1e12
    return {"status": "ok", "images_per_s": round(img_s, 1), "step_ms": round(1000 * dt / ITERS, 1),
            "peak_alloc_gb": peak_alloc, "peak_reserved_gb": peak_res, **gpu,
            "images_per_joule": round(img_s / gpu["power_avg_w"], 2) if gpu["power_avg_w"] else None,
            "approx_tflops": round(tflops, 1), "approx_mfu_pct": round(100 * tflops / PEAK_TFLOPS[prec], 1)}

NAME_RN = "train_resnet50"
done = done_keys(NAME_RN)
for prec in PRECISIONS:
    for bs in RN_BATCHES:
        key = f"resnet50|{prec}|bs{bs}|img{IMG}"
        if key in done:
            print("skip (done):", key); continue
        row = base_row(key, model="resnet50", task="train", precision=prec, batch=bs, img_size=IMG,
                       warmup=WARMUP, iters=ITERS, optimizer="SGD(momentum)", data="synthetic",
                       memory_format="channels_last")
        try:
            row.update(resnet_train(prec, bs))
        except torch.cuda.OutOfMemoryError:
            row.update(status="OOM")
        cleanup()
        save_result(NAME_RN, row)
        print(f"{key:28s} {row['status']:4s} {row.get('images_per_s', '-'):>8} img/s  "
              f"peak {row.get('peak_alloc_gb', '-')} GB  {row.get('power_avg_w', '-')} W")

df = pd.read_csv(f"{RESULTS_DIR}/{NAME_RN}.csv")
print(df[["precision", "batch", "status", "images_per_s", "step_ms", "peak_alloc_gb",
          "power_avg_w", "images_per_joule", "approx_mfu_pct"]].to_string(index=False))


In [ ]:
# Block 5 — Qwen2.5-1.5B training (random init from config, bf16 AMP, fp32 master weights, fused AdamW)
from transformers import AutoConfig, AutoModelForCausalLM

QWEN = "Qwen/Qwen2.5-1.5B"
qcfg = AutoConfig.from_pretrained(QWEN)      # config.json only, no weights
qcfg.use_cache = False
Q_WARMUP, Q_ITERS = 5, 20
Q_CONFIGS = [            # (batch, seq_len, grad_checkpointing)
    (4, 512, True),
    (8, 512, True),
    (4, 1024, True),
    (2, 512, False),     # no checkpointing -> shows the recompute cost
]

def qwen_train(bs, seq, ckpt, prec="bf16"):
    set_tf32(True)
    torch.manual_seed(0)
    with torch.device("cuda"):
        model = AutoModelForCausalLM.from_config(qcfg, attn_implementation="sdpa")
    model.train()
    if ckpt:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    n_params = sum(p.numel() for p in model.parameters())
    opt = torch.optim.AdamW(model.parameters(), lr=1e-5, fused=True)
    x = torch.randint(0, qcfg.vocab_size, (bs, seq), device="cuda")

    def step():
        with amp_ctx(prec):
            loss = model(input_ids=x, labels=x).loss
        loss.backward(); opt.step(); opt.zero_grad(set_to_none=True)

    dt, gpu, peak_alloc, peak_res = run_timed(step, Q_WARMUP, Q_ITERS)
    tok_s = bs * seq * Q_ITERS / dt
    tflops = 6 * n_params * tok_s / 1e12          # model FLOPs only (recompute not counted)
    return {"status": "ok", "tokens_per_s": round(tok_s), "step_ms": round(1000 * dt / Q_ITERS, 1),
            "peak_alloc_gb": peak_alloc, "peak_reserved_gb": peak_res, **gpu,
            "tokens_per_joule": round(tok_s / gpu["power_avg_w"], 1) if gpu["power_avg_w"] else None,
            "approx_tflops": round(tflops, 1), "approx_mfu_pct": round(100 * tflops / PEAK_TFLOPS[prec], 1),
            "n_params_m": round(n_params / 1e6, 1), "attn_impl": model.config._attn_implementation}

NAME_Q = "train_qwen2.5-1.5b"
done = done_keys(NAME_Q)
for bs, seq, ckpt in Q_CONFIGS:
    key = f"qwen2.5-1.5b|bf16|bs{bs}|seq{seq}|ckpt{int(ckpt)}"
    if key in done:
        print("skip (done):", key); continue
    row = base_row(key, model="qwen2.5-1.5b", task="train", precision="bf16", batch=bs, seq_len=seq,
                   grad_ckpt=ckpt, warmup=Q_WARMUP, iters=Q_ITERS, optimizer="AdamW(fused)",
                   master_weights="fp32", data="synthetic", init="random(from config)")
    try:
        row.update(qwen_train(bs, seq, ckpt))
    except torch.cuda.OutOfMemoryError:
        row.update(status="OOM")
    cleanup()
    save_result(NAME_Q, row)
    print(f"{key:38s} {row['status']:4s} {row.get('tokens_per_s', '-'):>7} tok/s  "
          f"peak {row.get('peak_alloc_gb', '-')} GB  {row.get('power_avg_w', '-')} W")

df = pd.read_csv(f"{RESULTS_DIR}/{NAME_Q}.csv")
print(df[["batch", "seq_len", "grad_ckpt", "status", "tokens_per_s", "step_ms", "peak_alloc_gb",
          "power_avg_w", "tokens_per_joule", "approx_mfu_pct"]].to_string(index=False))


In [ ]:
# Block 5b — verify Qwen param dtype, relabel Block 5 rows, re-run with true fp32 master weights
with torch.device("meta"):
    _m = AutoModelForCausalLM.from_config(qcfg)
default_dtype = next(_m.parameters()).dtype; del _m
print("from_config default param dtype:", default_dtype)

def rewrite_results(name, rows):
    tmp = _path(name) + ".tmp"
    with open(tmp, "w") as f:
        for r in rows: f.write(json.dumps(r) + "\n")
        f.flush(); os.fsync(f.fileno())
    os.replace(tmp, _path(name))
    cols = list(dict.fromkeys(k for r in rows for k in r))
    with open(f"{RESULTS_DIR}/{name}.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols); w.writeheader(); w.writerows(rows)

rows = [json.loads(l) for l in open(_path(NAME_Q)) if l.strip()]
stale = [r for r in rows if "param_dtype" not in r]          # rows written by Block 5
if stale:
    dt_name = str(default_dtype).replace("torch.", "")
    for r in stale:
        r["param_dtype"] = dt_name
        if default_dtype == torch.bfloat16:
            r.update(key=r["key"].replace("|bf16|", "|bf16-pure|"), precision="bf16-pure",
                     master_weights="none (bf16 weights + bf16 AdamW states)")
    rewrite_results(NAME_Q, rows)
    print(f"tagged {len(stale)} Block 5 rows with param_dtype={dt_name}")

def qwen_train_fp32master(bs, seq, ckpt, prec="bf16"):
    set_tf32(True)
    torch.manual_seed(0)
    with torch.device("cuda"):
        model = AutoModelForCausalLM.from_config(qcfg, attn_implementation="sdpa")
    model = model.float().train()                              # force fp32 master weights
    assert next(model.parameters()).dtype == torch.float32
    if ckpt:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    n_params = sum(p.numel() for p in model.parameters())
    opt = torch.optim.AdamW(model.parameters(), lr=1e-5, fused=True)
    x = torch.randint(0, qcfg.vocab_size, (bs, seq), device="cuda")

    def step():
        with amp_ctx(prec):
            loss = model(input_ids=x, labels=x).loss
        loss.backward(); opt.step(); opt.zero_grad(set_to_none=True)

    dt, gpu, peak_alloc, peak_res = run_timed(step, Q_WARMUP, Q_ITERS)
    tok_s = bs * seq * Q_ITERS / dt
    tflops = 6 * n_params * tok_s / 1e12
    return {"status": "ok", "tokens_per_s": round(tok_s), "step_ms": round(1000 * dt / Q_ITERS, 1),
            "peak_alloc_gb": peak_alloc, "peak_reserved_gb": peak_res, **gpu,
            "tokens_per_joule": round(tok_s / gpu["power_avg_w"], 1) if gpu["power_avg_w"] else None,
            "approx_tflops": round(tflops, 1), "approx_mfu_pct": round(100 * tflops / PEAK_TFLOPS[prec], 1),
            "n_params_m": round(n_params / 1e6, 1), "attn_impl": model.config._attn_implementation}

if default_dtype == torch.float32:
    print("Block 5 already used fp32 master weights -> no re-run needed")
else:
    done = done_keys(NAME_Q)
    for bs, seq, ckpt in Q_CONFIGS:
        key = f"qwen2.5-1.5b|bf16-amp|bs{bs}|seq{seq}|ckpt{int(ckpt)}"
        if key in done:
            print("skip (done):", key); continue
        row = base_row(key, model="qwen2.5-1.5b", task="train", precision="bf16", batch=bs, seq_len=seq,
                       grad_ckpt=ckpt, warmup=Q_WARMUP, iters=Q_ITERS, optimizer="AdamW(fused)",
                       master_weights="fp32", param_dtype="float32", data="synthetic",
                       init="random(from config)")
        try:
            row.update(qwen_train_fp32master(bs, seq, ckpt))
        except torch.cuda.OutOfMemoryError:
            row.update(status="OOM")
        cleanup()
        save_result(NAME_Q, row)
        print(f"{key:38s} {row['status']:4s} {row.get('tokens_per_s', '-'):>7} tok/s  "
              f"peak {row.get('peak_alloc_gb', '-')} GB  {row.get('power_avg_w', '-')} W")

df = pd.read_csv(f"{RESULTS_DIR}/{NAME_Q}.csv")
print(df[["precision", "master_weights", "batch", "seq_len", "grad_ckpt", "status", "tokens_per_s",
          "step_ms", "peak_alloc_gb", "power_avg_w", "tokens_per_joule", "approx_mfu_pct"]].to_string(index=False))


In [ ]:
# Block 6 — HF transformers generate() baseline (Qwen2.5-1.5B-Instruct, bf16, greedy, fixed lengths)
from transformers import AutoTokenizer
import statistics

INF_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
IN_LEN, OUT_LEN = 128, 256
HF_BATCHES = [1, 8, 32]
REPEATS = 2

set_tf32(True)
tok = AutoTokenizer.from_pretrained(INF_MODEL)
lm = AutoModelForCausalLM.from_pretrained(INF_MODEL, dtype=torch.bfloat16, attn_implementation="sdpa",
                                          device_map="cuda").eval()
text = "The history of high-performance computing is a story of parallelism and memory bandwidth. " * 64
base_ids = tok(text, return_tensors="pt").input_ids[0, :IN_LEN]
assert base_ids.numel() == IN_LEN

@torch.inference_mode()
def gen(ids, n_new):
    return lm.generate(input_ids=ids, attention_mask=torch.ones_like(ids), max_new_tokens=n_new,
                       min_new_tokens=n_new, do_sample=False, pad_token_id=tok.eos_token_id)

def timed(fn):
    t0 = time.perf_counter(); out = fn(); torch.cuda.synchronize()
    return time.perf_counter() - t0, out

def hf_bench(bs):
    ids = base_ids.repeat(bs, 1).cuda()
    gen(ids, 8); torch.cuda.synchronize()                                   # warmup
    ttft = statistics.median(timed(lambda: gen(ids, 1))[0] for _ in range(3))
    torch.cuda.reset_peak_memory_stats()
    times = []
    with GPUSampler() as s:
        for _ in range(REPEATS):
            t, out = timed(lambda: gen(ids, OUT_LEN)); times.append(t)
    assert out.shape[1] == IN_LEN + OUT_LEN, out.shape
    t, gpu = statistics.median(times), s.stats()
    out_tok_s = bs * OUT_LEN / t
    decode_tok_s = bs * (OUT_LEN - 1) / (t - ttft)
    return {"status": "ok", "ttft_ms": round(1000 * ttft, 1), "e2e_s": round(t, 2),
            "output_tokens_per_s": round(out_tok_s, 1), "decode_tokens_per_s": round(decode_tok_s, 1),
            "per_seq_decode_tok_s": round(decode_tok_s / bs, 1),
            "peak_alloc_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2), **gpu,
            "tokens_per_joule": round(out_tok_s / gpu["power_avg_w"], 2) if gpu["power_avg_w"] else None}

NAME_HF = "infer_hf"
done = done_keys(NAME_HF)
for bs in HF_BATCHES:
    key = f"hf-generate|qwen2.5-1.5b-instruct|bf16|bs{bs}|in{IN_LEN}|out{OUT_LEN}"
    if key in done:
        print("skip (done):", key); continue
    row = base_row(key, engine="hf-transformers", model="qwen2.5-1.5b-instruct", task="inference",
                   precision="bf16", batch=bs, input_len=IN_LEN, output_len=OUT_LEN, decoding="greedy",
                   attn_impl="sdpa", transformers=transformers.__version__)
    try:
        row.update(hf_bench(bs))
    except torch.cuda.OutOfMemoryError:
        row.update(status="OOM")
    cleanup(); save_result(NAME_HF, row)
    print(f"bs{bs:<3} {row['status']:4s} TTFT {row.get('ttft_ms', '-')} ms  "
          f"{row.get('output_tokens_per_s', '-')} out tok/s  ({row.get('per_seq_decode_tok_s', '-')} tok/s/seq)  "
          f"peak {row.get('peak_alloc_gb', '-')} GiB  {row.get('power_avg_w', '-')} W")

df = pd.read_csv(f"{RESULTS_DIR}/{NAME_HF}.csv")
print(df[["batch", "status", "ttft_ms", "e2e_s", "output_tokens_per_s", "per_seq_decode_tok_s",
          "peak_alloc_gb", "gpu_util_avg", "power_avg_w", "tokens_per_joule"]].to_string(index=False))

del lm; cleanup()          # free the GPU before vLLM


## Isolated vLLM inference

The recorded run used vLLM 0.31.0, torch 2.13.0+cu132, `gpu_memory_utilization=0.6`, and the sampler setting below. Package availability and future driver compatibility need validation on the target runtime.


In [ ]:
%pip install -q uv
!uv venv /content/vllm-env --python 3.12
!uv pip install --python /content/vllm-env/bin/python vllm==0.31.0 torch==2.13.0 nvidia-ml-py ninja --torch-backend=auto


In [ ]:
%%writefile /content/vllm_bench.py
# A100 offline reproduction: one-token batch duration is a proxy, NOT request TTFT.
# No corrected physical-energy claim; A100 scientific-v2 host validation pending.
import json, os, sys, time, random, statistics, threading, datetime
import pynvml, torch, vllm
from vllm import LLM, SamplingParams
from vllm.inputs import TokensPrompt

RESULTS_DIR = sys.argv[1]
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
IN_LEN, OUT_LEN = 128, 256
NUM_PROMPTS = [1, 8, 32, 128, 512]
GPU_UTIL = float(sys.argv[2]) if len(sys.argv) > 2 else 0.85
OUT = f"{RESULTS_DIR}/infer_vllm.jsonl"

pynvml.nvmlInit()
_H = pynvml.nvmlDeviceGetHandleByIndex(0)

class GPUSampler:
    def __init__(self, interval=0.1):
        self.interval, self.util, self.power, self.mem = interval, [], [], []
        self._stop = threading.Event()
    def _run(self):
        while not self._stop.is_set():
            self.util.append(pynvml.nvmlDeviceGetUtilizationRates(_H).gpu)
            self.power.append(pynvml.nvmlDeviceGetPowerUsage(_H) / 1000)
            self.mem.append(pynvml.nvmlDeviceGetMemoryInfo(_H).used / 2**30)
            time.sleep(self.interval)
    def __enter__(self):
        self._t = threading.Thread(target=self._run, daemon=True); self._t.start(); return self
    def __exit__(self, *a):
        self._stop.set(); self._t.join()
    def stats(self):
        avg = lambda x: round(sum(x) / len(x), 1) if x else None
        return {"gpu_util_avg": avg(self.util), "gpu_util_max": max(self.util, default=None),
                "power_avg_w": avg(self.power), "power_max_w": round(max(self.power, default=0), 1),
                "nvml_mem_max_gb": round(max(self.mem, default=0), 2)}

def key(n): return f"vllm|qwen2.5-1.5b-instruct|bf16|n{n}|in{IN_LEN}|out{OUT_LEN}"

def main():
    done = set()
    if os.path.exists(OUT):
        done = {json.loads(l)["key"] for l in open(OUT) if l.strip()}
    todo = [n for n in NUM_PROMPTS if key(n) not in done]
    if not todo:
        print("all vLLM configs already done"); return

    t0 = time.perf_counter()
    llm = LLM(model=MODEL, dtype="bfloat16", gpu_memory_utilization=GPU_UTIL, max_model_len=1024,
              enable_prefix_caching=False, seed=0)
    load_s = time.perf_counter() - t0

    rng = random.Random(0)
    def prompts(n):   # distinct random-token prompts of exactly IN_LEN tokens
        return [TokensPrompt(prompt_token_ids=[rng.randrange(1000, 150000) for _ in range(IN_LEN)])
                for _ in range(n)]
    sp_full = SamplingParams(temperature=0.0, max_tokens=OUT_LEN, min_tokens=OUT_LEN, ignore_eos=True)
    sp_one = SamplingParams(temperature=0.0, max_tokens=1)

    llm.generate(prompts(8), sp_full, use_tqdm=False)          # warmup

    for n in todo:
        ps = prompts(n)
        ttfts = []
        for _ in range(3):
            t = time.perf_counter(); llm.generate(ps, sp_one, use_tqdm=False)
            ttfts.append(time.perf_counter() - t)
        ttft = statistics.median(ttfts)
        times = []
        with GPUSampler() as s:
            for _ in range(2):
                t = time.perf_counter(); outs = llm.generate(ps, sp_full, use_tqdm=False)
                times.append(time.perf_counter() - t)
        assert all(len(o.outputs[0].token_ids) == OUT_LEN for o in outs)
        t, gpu = statistics.median(times), s.stats()
        out_tok_s = n * OUT_LEN / t
        row = {"key": key(n), "gpu": torch.cuda.get_device_name(0), "engine": "vllm",
               "vllm": vllm.__version__, "torch": torch.__version__,
               "ts": datetime.datetime.now().isoformat(timespec="seconds"),
               "model": "qwen2.5-1.5b-instruct", "task": "inference", "precision": "bf16",
               "batch": n, "input_len": IN_LEN, "output_len": OUT_LEN, "decoding": "greedy(ignore_eos)",
               "gpu_memory_utilization": GPU_UTIL, "prefix_caching": False, "load_s": round(load_s, 1),
               "status": "ok", "one_token_batch_completion_proxy_ms": round(1000 * ttft, 1), "e2e_s": round(t, 2),
               "protocol_version": "a100-proxy-label-correction-not-validated",
               "one_token_call_times_s": ttfts, "full_batch_call_times_s": times,
               "true_request_ttft_measured": False,
               "output_tokens_per_s": round(out_tok_s, 1),
               "total_tokens_per_s": round(n * (IN_LEN + OUT_LEN) / t, 1),
               "per_seq_decode_proxy_tok_s": round((OUT_LEN - 1) / (t - ttft), 1) if t > ttft else None, **gpu,
               "tokens_per_joule": None,
               "energy_status": "not_validated; no aligned integrated energy recorded"}
        with open(OUT, "a") as f:
            f.write(json.dumps(row) + "\n"); f.flush(); os.fsync(f.fileno())
        print(f"n={n:<4} one-token-batch proxy {row['one_token_batch_completion_proxy_ms']} ms  "
              f"{row['output_tokens_per_s']} out tok/s  "
              f"({row['per_seq_decode_proxy_tok_s']} proxy tok/s/seq)  {gpu['power_avg_w']} W")

if __name__ == "__main__":
    main()


In [ ]:
cleanup()
VENV_PY = '/content/vllm-env/bin/python'
run_env = {**os.environ, 'PATH': '/content/vllm-env/bin:' + os.environ['PATH'],
           'VLLM_USE_FLASHINFER_SAMPLER': '0'}
log_path = f'{RESULTS_DIR}/vllm_log.txt'
with open(log_path, 'w') as run_log:
    result = subprocess.run([VENV_PY, '/content/vllm_bench.py', RESULTS_DIR, '0.6'],
                            stdout=run_log, stderr=subprocess.STDOUT, env=run_env)
print('vLLM exit code:', result.returncode, 'log:', log_path)
if result.returncode:
    raise RuntimeError(f'vLLM failed; inspect {log_path}')


In [ ]:
# Rebuild tables from the new local JSONL records.
names = ['train_gpt2', 'train_resnet50', 'train_qwen2.5-1.5b', 'infer_hf', 'infer_vllm']
tables = []
for name in names:
    with open(f'{RESULTS_DIR}/{name}.jsonl') as stream:
        records = [json.loads(line) for line in stream if line.strip()]
    frame = pd.DataFrame(records)
    frame.to_csv(f'{RESULTS_DIR}/{name}.csv', index=False)
    tables.append(frame.assign(benchmark=name))
pd.concat(tables, ignore_index=True).to_csv(f'{RESULTS_DIR}/all_results.csv', index=False)
print('Reproduction outputs:', RESULTS_DIR, sorted(os.listdir(RESULTS_DIR)))
